In [ ]:
import os

os.environ["KAGGLE_API_TOKEN"] = "KGAT_e6b85efef062074c2698fc4104fd7663"


In [ ]:
import os

print(os.listdir())

In [ ]:
from google.colab import files

files.download("pneumonia_resnet18_model.pth")

In [ ]:
!pip -q install gradio

In [ ]:
import gradio as gr
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from PIL import Image

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

app_model = models.resnet18(weights=None)
num_features = app_model.fc.in_features
app_model.fc = nn.Linear(num_features, 2)

app_model.load_state_dict(torch.load("pneumonia_resnet18_model (1).pth", map_location=device))
app_model = app_model.to(device)
app_model.eval()

print(" Trained model loaded and ready for the app")
print("Using device:", device)

In [ ]:
# Image preprocessing (must match training!)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

classes = ["NORMAL", "PNEUMONIA"]

def predict_xray(image):

    # Convert uploaded image to RGB
    image = image.convert("RGB")

    # Apply preprocessing
    image_tensor = transform(image).unsqueeze(0).to(device)

    # Make prediction
    with torch.no_grad():
        outputs = app_model(image_tensor)
        probabilities = F.softmax(outputs, dim=1)

    confidence, predicted = torch.max(probabilities, 1)

    prediction = classes[predicted.item()]
    confidence = confidence.item() * 100

    return f"{prediction} ({confidence:.2f}% confidence)"

In [ ]:
interface = gr.Interface(
    fn=predict_xray,
    inputs=gr.Image(type="pil"),
    outputs=gr.Textbox(label="Prediction"),
    title=" Pneumonia Detection AI",
    description="""
Upload a chest X-ray image.

This AI was trained using transfer learning with ResNet18 to classify chest X-rays as either NORMAL or PNEUMONIA.

 This project is for educational and research purposes only and should not be used for medical diagnosis.
"""
)

In [ ]:
interface.launch(share=True)

In [ ]:
from matplotlib import pyplot as plt
import numpy as np

def predict_xray_with_gradcam(image):

    image = image.convert("RGB")

    input_tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = app_model(input_tensor)
        probabilities = F.softmax(outputs, dim=1)

    confidence, predicted = torch.max(probabilities, 1)

    prediction = classes[predicted.item()]
    confidence_score = confidence.item() * 100

    normal_prob = probabilities[0][0].item() * 100
    pneumonia_prob = probabilities[0][1].item() * 100

    target_layers = [app_model.layer4[-1]]

    cam = GradCAM(
        model=app_model,
        target_layers=target_layers
    )

    grayscale_cam = cam(input_tensor=input_tensor)[0]

    img = image.resize((224,224))
    img = np.array(img).astype(np.float32)/255.0

    visualization = show_cam_on_image(
        img,
        grayscale_cam,
        use_rgb=True
    )

    result = f"""
 PREDICTION
────────────────────────────

{prediction}

 CONFIDENCE

{confidence_score:.2f}%

────────────────────────────

 CLASS PROBABILITIES

NORMAL:      {normal_prob:.2f}%

PNEUMONIA:   {pneumonia_prob:.2f}%

────────────────────────────

 HOW THE AI MADE THIS DECISION

• Image resized to 224×224 pixels

• Normalized using ImageNet statistics

• Processed with a ResNet18 convolutional neural network

• Grad-CAM highlights image regions that most influenced the prediction.

⚠ Highlighted regions represent model attention, NOT confirmed disease.

────────────────────────────

⚙ MODEL

Architecture: ResNet18

Training Images: 5,216

Test Accuracy: 83.8%

Framework: PyTorch

Explainability: Grad-CAM

────────────────────────────

⚠ Educational Research Demonstration

Not intended for medical diagnosis.
"""

    return result, visualization

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import tempfile

def professional_xray_analysis(image):
    image = image.convert("RGB")

    input_tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = app_model(input_tensor)
        probabilities = F.softmax(outputs, dim=1)

    normal_prob = probabilities[0][0].item() * 100
    pneumonia_prob = probabilities[0][1].item() * 100

    confidence, predicted = torch.max(probabilities, 1)
    prediction = classes[predicted.item()]
    confidence_score = confidence.item() * 100

    target_layers = [app_model.layer4[-1]]
    cam = GradCAM(model=app_model, target_layers=target_layers)
    grayscale_cam = cam(input_tensor=input_tensor)[0]

    resized_img = image.resize((224, 224))
    img_array = np.array(resized_img).astype(np.float32) / 255.0
    heatmap = show_cam_on_image(img_array, grayscale_cam, use_rgb=True)

    prob_df = pd.DataFrame({
        "Class": ["NORMAL", "PNEUMONIA"],
        "Probability (%)": [normal_prob, pneumonia_prob]
    })

    explanation = f"""
## Prediction Result

**Prediction:** {prediction}
**Confidence:** {confidence_score:.2f}%

---

## Class Probability Breakdown

- **NORMAL:** {normal_prob:.2f}%
- **PNEUMONIA:** {pneumonia_prob:.2f}%

---

##  Grad-CAM Interpretation

The heatmap shows which regions of the X-ray most influenced the model's prediction.

Red/yellow regions had stronger influence.
Blue/green regions had weaker influence.

Important: Grad-CAM does **not** prove disease location. It only shows model attention.

---

##  Model Information

- **Architecture:** ResNet18
- **Method:** Transfer Learning
- **Framework:** PyTorch
- **Training Images:** 5,216
- **Test Accuracy:** 83.81%
- **Explainability Tool:** Grad-CAM

---

## Disclaimer

This is an educational and research demonstration only. It is **not** intended for medical diagnosis.
"""

    return explanation, heatmap, prob_df

In [ ]:
final_app = gr.Interface(
    fn=professional_xray_analysis,

    inputs=gr.Image(
        type="pil",
        label="Upload Chest Radiograph"
    ),

    outputs=[
        gr.Markdown(label="Analysis Report"),
        gr.Image(label="Grad-CAM Attention Map"),
        gr.Dataframe(label="Prediction Probabilities")
    ],

    title="Explainable Deep Learning for Pneumonia Detection",

    description="""
Research demonstration using a ResNet18 convolutional neural network with Grad-CAM interpretability.

Upload a posterior-anterior chest radiograph to generate a prediction, confidence score,
class probability estimates, and an attention map illustrating the image regions that
most influenced the model's prediction.

This application is intended solely for educational and research purposes and should
not be used for clinical diagnosis or medical decision-making.
"""
)

final_app.launch(share=True)

In [ ]:
with gr.Blocks(title="Explainable Deep Learning for Pneumonia Detection") as research_app:

    gr.Markdown("""
# Explainable Deep Learning for Pneumonia Detection

Research demonstration using ResNet18 transfer learning and Grad-CAM interpretability.

This application is intended for educational and research purposes only. It is not intended for clinical diagnosis or medical decision-making.
""")

    with gr.Row():
        with gr.Column():
            input_image = gr.Image(type="pil", label="Upload Chest Radiograph")
            submit_btn = gr.Button("Run Analysis")

        with gr.Column():
            report_output = gr.Markdown(label="Analysis Report")
            heatmap_output = gr.Image(label="Grad-CAM Attention Map")
            prob_output = gr.Dataframe(label="Prediction Probabilities")

    submit_btn.click(
        fn=professional_xray_analysis,
        inputs=input_image,
        outputs=[report_output, heatmap_output, prob_output]
    )

research_app.launch(share=True)

In [ ]:
import gradio as gr

In [ ]:
gradcam_interface = gr.Interface(
    fn=predict_xray_with_gradcam,
    inputs=gr.Image(type="pil", label="Upload Chest X-ray"),
    outputs=[
        gr.Textbox(label="AI Prediction + Explanation"),
        gr.Image(label="Grad-CAM Heatmap")
    ],
    title=" Explainable Pneumonia Detection AI",
    description="""
Upload a chest X-ray image. The model predicts NORMAL or PNEUMONIA and displays a Grad-CAM heatmap showing which image regions influenced the prediction.

 Educational/research demo only — not for medical diagnosis.
"""
)

gradcam_interface.launch(share=True)

In [ ]:
!pip -q install kaggle

In [ ]:
!mkdir -p ~/.kaggle
!echo $KAGGLE_API_TOKEN > ~/.kaggle/access_token
!chmod 600 ~/.kaggle/access_token

In [ ]:
!kaggle datasets list -s "chest xray pneumonia"

In [ ]:
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia

In [ ]:
!unzip -q chest-xray-pneumonia.zip

In [ ]:
import os
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

train_dataset = datasets.ImageFolder("chest_xray/chest_xray/train", transform=transform)
val_dataset = datasets.ImageFolder("chest_xray/chest_xray/val", transform=transform)
test_dataset = datasets.ImageFolder("chest_xray/chest_xray/test", transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print("Device:", device)
print("Training images:", len(train_dataset))
print("Classes:", train_dataset.classes)
print("Model final layer:", model.fc)

In [ ]:
num_epochs = 3

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_loader)
    epoch_accuracy = 100 * correct / total

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Training Loss: {epoch_loss:.4f}")
    print(f"Training Accuracy: {epoch_accuracy:.2f}%")
    print("-" * 30)

In [ ]:
# Get one batch of images
images, labels = next(iter(train_loader))

# Move the batch to the GPU
images = images.to(device)
labels = labels.to(device)

# Let the model make predictions
outputs = model(images)

print("Output shape:", outputs.shape)
print(outputs[:5])

In [ ]:
loss = criterion(outputs, labels)

print("Loss:", loss.item())
print("First 10 true labels:", labels[:10])

In [ ]:
_, predicted = torch.max(outputs, 1)

print("First 10 predicted labels:", predicted[:10])
print("First 10 true labels:", labels[:10])

correct = (predicted == labels).sum().item()
print("Correct in this batch:", correct, "out of", labels.size(0))

In [ ]:
# Calculate gradients
optimizer.zero_grad()

# Compute the loss
loss = criterion(outputs, labels)

# Backpropagation
loss.backward()

# Update the model's weights
optimizer.step()

print(" One learning step completed!")

In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

test_accuracy = 100 * correct / total

print(f"Test Accuracy: {test_accuracy:.2f}%")

In [ ]:
print(os.listdir("chest_xray/chest_xray"))

In [ ]:
print(os.listdir("chest_xray/chest_xray/train"))

In [ ]:
normal_train = len(os.listdir("chest_xray/chest_xray/train/NORMAL"))
pneumonia_train = len(os.listdir("chest_xray/chest_xray/train/PNEUMONIA"))

print("Normal training images:", normal_train)
print("Pneumonia training images:", pneumonia_train)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os

# Path to one normal X-ray
image_path = os.path.join(
    "chest_xray",
    "chest_xray",
    "train",
    "NORMAL",
    os.listdir("chest_xray/chest_xray/train/NORMAL")[0]
)

# Open the image
image = Image.open(image_path)

# Display it
plt.imshow(image, cmap="gray")
plt.title("Normal Chest X-ray")
plt.axis("off")

plt.show()

In [ ]:
# Path to one pneumonia X-ray
image_path = os.path.join(
    "chest_xray",
    "chest_xray",
    "train",
    "PNEUMONIA",
    os.listdir("chest_xray/chest_xray/train/PNEUMONIA")[0]
)

# Open the image
image = Image.open(image_path)

# Display it
plt.imshow(image, cmap="gray")
plt.title("Pneumonia Chest X-ray")
plt.axis("off")

plt.show()

In [ ]:
from PIL import Image
import os

normal_path = os.path.join(
    "chest_xray",
    "chest_xray",
    "train",
    "NORMAL",
    os.listdir("chest_xray/chest_xray/train/NORMAL")[0]
)

pneumonia_path = os.path.join(
    "chest_xray",
    "chest_xray",
    "train",
    "PNEUMONIA",
    os.listdir("chest_xray/chest_xray/train/PNEUMONIA")[0]
)

normal_image = Image.open(normal_path)
pneumonia_image = Image.open(pneumonia_path)

print("Normal image size:", normal_image.size)
print("Pneumonia image size:", pneumonia_image.size)

In [ ]:
import torch
from torchvision import datasets, transforms

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

In [ ]:
# Apply the preprocessing transform to one normal image
transformed_image = transform(normal_image)

print("Tensor shape:", transformed_image.shape)
print("Data type:", transformed_image.dtype)

In [ ]:
train_dataset = datasets.ImageFolder(
    root="chest_xray/chest_xray/train",
    transform=transform
)

print("Number of training images:", len(train_dataset))

In [ ]:
print("Classes:", train_dataset.classes)
print("Class-to-index mapping:", train_dataset.class_to_idx)

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

print(train_loader)

In [ ]:
# Get one batch of images
images, labels = next(iter(train_loader))

print("Images shape:", images.shape)
print("Labels shape:", labels.shape)

print("\nFirst 10 labels:")
print(labels[:10])

In [ ]:
val_dataset = datasets.ImageFolder(
    root="chest_xray/chest_xray/val",
    transform=transform
)

test_dataset = datasets.ImageFolder(
    root="chest_xray/chest_xray/test",
    transform=transform
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

print("Validation images:", len(val_dataset))
print("Test images:", len(test_dataset))
print("Validation classes:", val_dataset.classes)
print("Test classes:", test_dataset.classes)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

In [ ]:
from torchvision import models
import torch.nn as nn

In [ ]:
# Load a pretrained ResNet18 model
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

print(model)

In [ ]:
# Replace the final classification layer for our 2 classes
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)

# Move model to GPU
model = model.to(device)

print(model.fc)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print("Loss function:", criterion)
print("Optimizer:", optimizer)

In [ ]:
import os

print(os.listdir())

In [ ]:
import os

print(os.listdir("chest_xray"))


In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

test_accuracy = 100 * correct / total

print(f"Test Accuracy: {test_accuracy:.2f}%")

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder("chest_xray/chest_xray/train", transform=transform)
val_dataset = datasets.ImageFolder("chest_xray/chest_xray/val", transform=transform)
test_dataset = datasets.ImageFolder("chest_xray/chest_xray/test", transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

print(" Normalized data + fresh model ready")

In [ ]:
num_epochs = 3

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_loader)
    epoch_accuracy = 100 * correct / total

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Training Loss: {epoch_loss:.4f}")
    print(f"Training Accuracy: {epoch_accuracy:.2f}%")
    print("-" * 30)

In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

test_accuracy = 100 * correct / total

print(f"Test Accuracy: {test_accuracy:.2f}%")

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

model.eval()

all_labels = []
all_predictions = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        all_labels.extend(labels.cpu().numpy())
        all_predictions.extend(predicted.cpu().numpy())

In [ ]:
# Build the confusion matrix
cm = confusion_matrix(all_labels, all_predictions)

# Display it
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["NORMAL", "PNEUMONIA"]
)

disp.plot(cmap="Blues")
plt.title("Confusion Matrix")
plt.show()

# Also print the raw numbers
print(cm)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    all_labels,
    all_predictions,
    target_names=["NORMAL", "PNEUMONIA"]
))

In [ ]:
import random
import torch.nn.functional as F

# Pick a random image from the test set
idx = random.randint(0, len(test_dataset) - 1)
image_tensor, true_label = test_dataset[idx]

# Add batch dimension and move to GPU
input_image = image_tensor.unsqueeze(0).to(device)

model.eval()
with torch.no_grad():
    output = model(input_image)
    probabilities = F.softmax(output, dim=1)
    confidence, predicted_label = torch.max(probabilities, 1)

predicted_class = test_dataset.classes[predicted_label.item()]
true_class = test_dataset.classes[true_label]

print("Actual Label:", true_class)
print("Predicted Label:", predicted_class)
print(f"Confidence: {confidence.item() * 100:.2f}%")

if predicted_class == true_class:
    print(" Correct prediction")
else:
    print(" Incorrect prediction")

In [ ]:
!pip -q install grad-cam

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
import numpy as np
import cv2

In [ ]:
import random
from PIL import Image

# Pick a random test image
idx = random.randint(0, len(test_dataset) - 1)

# Get image tensor and true label
image_tensor, true_label = test_dataset[idx]

# Add batch dimension and move to GPU
input_tensor = image_tensor.unsqueeze(0).to(device)

print("Selected image index:", idx)
print("True label:", test_dataset.classes[true_label])
print("Input tensor shape:", input_tensor.shape)

In [ ]:
# Choose the final convolutional layer of ResNet18
target_layers = [model.layer4[-1]]

# Create Grad-CAM object
cam = GradCAM(
    model=model,
    target_layers=target_layers
)

# Generate Grad-CAM heatmap
grayscale_cam = cam(input_tensor=input_tensor)[0]

print("Grad-CAM heatmap created.")
print("Heatmap shape:", grayscale_cam.shape)


In [ ]:
# Convert tensor back into image format for visualization
img = image_tensor.permute(1, 2, 0).cpu().numpy()

# Undo ImageNet normalization so the image displays correctly
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])
img = std * img + mean
img = np.clip(img, 0, 1)

# Overlay Grad-CAM heatmap on image
visualization = show_cam_on_image(img, grayscale_cam, use_rgb=True)

plt.figure(figsize=(6, 6))
plt.imshow(visualization)
plt.title("Grad-CAM Heatmap")
plt.axis("off")
plt.show()

In [ ]:
import random
import matplotlib.pyplot as plt
import torch.nn.functional as F

model.eval()

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for ax in axes.flat:

    idx = random.randint(0, len(test_dataset)-1)

    image, label = test_dataset[idx]

    input_tensor = image.unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(input_tensor)
        probabilities = F.softmax(output, dim=1)

    confidence, prediction = torch.max(probabilities, 1)

    predicted_class = test_dataset.classes[prediction.item()]
    true_class = test_dataset.classes[label]

    # Undo normalization
    img = image.permute(1,2,0).numpy()
    mean = np.array([0.485,0.456,0.406])
    std = np.array([0.229,0.224,0.225])
    img = std * img + mean
    img = np.clip(img,0,1)

    ax.imshow(img)

    color = "green" if predicted_class == true_class else "red"

    ax.set_title(
        f"Actual: {true_class}\n"
        f"Pred: {predicted_class}\n"
        f"{confidence.item()*100:.1f}%",
        color=color,
        fontsize=10
    )

    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
wrong_examples = []

model.eval()

with torch.no_grad():
    for idx in range(len(test_dataset)):
        image_tensor, true_label = test_dataset[idx]
        input_tensor = image_tensor.unsqueeze(0).to(device)

        output = model(input_tensor)
        probabilities = F.softmax(output, dim=1)
        confidence, predicted_label = torch.max(probabilities, 1)

        if predicted_label.item() != true_label:
            wrong_examples.append({
                "idx": idx,
                "image_tensor": image_tensor,
                "true_label": true_label,
                "predicted_label": predicted_label.item(),
                "confidence": confidence.item()
            })

print("Number of wrong predictions:", len(wrong_examples))
print("First wrong example index:", wrong_examples[0]["idx"])
print("Actual:", test_dataset.classes[wrong_examples[0]["true_label"]])
print("Predicted:", test_dataset.classes[wrong_examples[0]["predicted_label"]])
print(f"Confidence: {wrong_examples[0]['confidence'] * 100:.2f}%")

In [ ]:
wrong = wrong_examples[0]

image_tensor = wrong["image_tensor"]
input_tensor = image_tensor.unsqueeze(0).to(device)

target_layers = [model.layer4[-1]]

cam = GradCAM(
    model=model,
    target_layers=target_layers
)

grayscale_cam = cam(input_tensor=input_tensor)[0]

img = image_tensor.permute(1, 2, 0).cpu().numpy()

mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])
img = std * img + mean
img = np.clip(img, 0, 1)

visualization = show_cam_on_image(img, grayscale_cam, use_rgb=True)

plt.figure(figsize=(6, 6))
plt.imshow(visualization)
plt.title(
    f"WRONG Prediction\n"
    f"Actual: {test_dataset.classes[wrong['true_label']]}\n"
    f"Predicted: {test_dataset.classes[wrong['predicted_label']]}\n"
    f"Confidence: {wrong['confidence']*100:.2f}%"
)
plt.axis("off")
plt.show()

In [ ]:
torch.save(model.state_dict(), "pneumonia_resnet18_model.pth")

print(" Model saved as pneumonia_resnet18_model.pth")

FINAL CLEAN APPLICATION